# 📄 PDF Ingestion → Local Qdrant via ngrok

Offloads heavy PDF parsing (Docling) and embedding (Qwen dense + BM25 sparse) to Colab's **free T4 GPU**, then upserts directly into your **local Qdrant** instance through an ngrok tunnel.

### Prerequisites
1. Start local Qdrant: `docker compose up qdrant -d`
2. Start ngrok: `ngrok http 6333`
3. Copy the public ngrok URL into `QDRANT_URL` below
4. Set your OpenRouter API key

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# Install all required packages (mirrors pyproject.toml)
!pip install -q \
    "qdrant-client>=1.17.0" \
    "fastembed>=0.7.4" \
    "docling>=2.75.0" \
    "langchain-openai>=1.1.10" \
    "huggingface-hub[hf-xet]>=0.36.2" \
    httpx \
    pydantic-settings

print("✅ Dependencies installed")

## 🔧 Cell 2 — Configuration

Set your ngrok public URL and API keys here.

In [ ]:
import os

# ── ngrok public URL pointing to your local Qdrant (port 6333) ──────────────
# Replace with the URL shown by: ngrok http 6333
QDRANT_URL = "https://abc123.ngrok-free.app"   # <-- CHANGE THIS
QDRANT_API_KEY = None                           # set if you configured one

# ── OpenRouter (dense embeddings via Qwen3-Embedding-8B) ────────────────────
OPENROUTER_API_KEY = "sk-or-..."               # <-- CHANGE THIS
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
EMBEDDING_MODEL = "qwen/qwen3-embedding-8b"

# ── Qdrant collection ────────────────────────────────────────────────────────
COLLECTION_NAME = "admin_docs"

# ── Ingestion tuning ─────────────────────────────────────────────────────────
CHUNK_MAX_TOKENS = 512
UPSERT_BATCH_SIZE = 64   # chunks per embed+upsert call
SLEEP_BETWEEN_BATCHES = 1.0  # seconds — stay under ngrok free-tier 40 req/min

# Push config into env so any library that reads os.environ picks it up
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print(f"✅ Config loaded")
print(f"   Qdrant  → {QDRANT_URL}")
print(f"   Embed   → {EMBEDDING_MODEL}")
print(f"   Collection: {COLLECTION_NAME}")

## 📦 Cell 3 — Pipeline Helpers

Inlined from `app/ingestion/chunker.py`, `app/services/embeddings.py`, and `app/services/vectorstore.py` — **no project imports needed**.

In [ ]:
"""─────────────────────────────────────────────────────────────────
Helper utilities — mirrors app/utils/helpers.py
─────────────────────────────────────────────────────────────────"""
from __future__ import annotations
import hashlib, uuid

def generate_doc_id(filename: str) -> str:
    """Deterministic SHA-256 document ID (first 16 hex chars)."""
    return hashlib.sha256(filename.encode()).hexdigest()[:16]

def generate_point_id() -> str:
    """Random UUID string for Qdrant point IDs."""
    return str(uuid.uuid4())

print("✅ Helper utilities ready")

In [ ]:
"""─────────────────────────────────────────────────────────────────
Parser — mirrors app/ingestion/parser.py
Uses do_ocr=False, do_table_structure=False, page_batch_size=3
for faster processing on Colab T4 GPU.
─────────────────────────────────────────────────────────────────"""
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

def parse_document(file_path: str | Path):
    """Convert a PDF to a DoclingDocument (GPU-optimised settings)."""
    file_path = Path(file_path)
    print(f"  📄 Parsing: {file_path.name}")

    pipeline_options = PdfPipelineOptions(
        do_ocr=False,                # skip OCR — faster, no GPU needed for text PDFs
        do_table_structure=False,    # skip table detection — saves GPU memory
        page_batch_size=3,           # process 3 pages at a time
    )

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    result = converter.convert(str(file_path))
    print(f"  ✅ Parsed '{file_path.name}' successfully")
    return result.document

print("✅ Parser ready")

In [ ]:
"""─────────────────────────────────────────────────────────────────
Chunker — mirrors app/ingestion/chunker.py
─────────────────────────────────────────────────────────────────"""
from __future__ import annotations
from dataclasses import dataclass, field
from docling.chunking import HybridChunker

@dataclass
class Chunk:
    """A single text chunk with metadata."""
    text: str
    headings: list[str]
    chunk_index: int
    page: int | None = field(default=None)


def _extract_page(rc) -> int | None:
    """Extract the first page number from a Docling chunk's provenance."""
    try:
        if not (hasattr(rc, "meta") and rc.meta and hasattr(rc.meta, "doc_items")):
            return None
        pages: list[int] = []
        for item in rc.meta.doc_items:
            if hasattr(item, "prov") and item.prov:
                for prov in item.prov:
                    if hasattr(prov, "page_no") and prov.page_no is not None:
                        pages.append(prov.page_no)
        return min(pages) if pages else None
    except Exception:
        return None


def chunk_document(doc, max_tokens: int = CHUNK_MAX_TOKENS) -> list[Chunk]:
    """Split a DoclingDocument into sized text chunks."""
    chunker = HybridChunker(
        tokenizer="sentence-transformers/all-MiniLM-L6-v2",
        max_tokens=max_tokens,
        merge_peers=True,
    )
    raw_chunks = list(chunker.chunk(doc))
    chunks: list[Chunk] = []
    for idx, rc in enumerate(raw_chunks):
        text = rc.text.strip()
        if not text:
            continue
        headings: list[str] = []
        if hasattr(rc, "meta") and rc.meta and hasattr(rc.meta, "headings"):
            headings = list(rc.meta.headings) if rc.meta.headings else []
        page = _extract_page(rc)
        chunks.append(Chunk(text=text, headings=headings, chunk_index=idx, page=page))

    print(f"  ✂️  {len(chunks)} chunks produced (max_tokens={max_tokens})")
    return chunks

print("✅ Chunker ready")

In [ ]:
"""─────────────────────────────────────────────────────────────────
Embeddings — mirrors app/services/embeddings.py
Calls OpenRouter's /embeddings endpoint (Qwen3-Embedding-8B).
─────────────────────────────────────────────────────────────────"""
from __future__ import annotations
import httpx

async def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts via OpenRouter and return dense vectors."""
    if not texts:
        return []
    url = f"{OPENROUTER_BASE_URL}/embeddings"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {"model": EMBEDDING_MODEL, "input": texts}
    async with httpx.AsyncClient(timeout=120.0) as client:
        response = await client.post(url, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()
    embeddings: list[list[float]] = [
        item["embedding"]
        for item in sorted(data["data"], key=lambda x: x["index"])
    ]
    return embeddings

print("✅ Embedding client ready")

In [ ]:
"""─────────────────────────────────────────────────────────────────
Vectorstore — mirrors app/services/vectorstore.py
AsyncQdrantClient + fastembed BM25 sparse encoder.
Connects via ngrok URL → local Qdrant.
─────────────────────────────────────────────────────────────────"""
from __future__ import annotations
from typing import Any
from fastembed import SparseTextEmbedding
from qdrant_client import AsyncQdrantClient, models

# ── Lazy singletons ────────────────────────────────────────────────────────
_qdrant_client: AsyncQdrantClient | None = None
_bm25_encoder: SparseTextEmbedding | None = None


def _get_bm25_encoder() -> SparseTextEmbedding:
    global _bm25_encoder
    if _bm25_encoder is None:
        print("  📥 Loading BM25 sparse encoder (Qdrant/bm25)…")
        _bm25_encoder = SparseTextEmbedding(model_name="Qdrant/bm25")
        print("  ✅ BM25 encoder loaded")
    return _bm25_encoder


def _encode_sparse(texts: list[str]) -> list[models.SparseVector]:
    """Encode texts into BM25 sparse vectors."""
    encoder = _get_bm25_encoder()
    results: list[models.SparseVector] = []
    for embedding in encoder.embed(texts):
        results.append(
            models.SparseVector(
                indices=embedding.indices.tolist(),
                values=embedding.values.tolist(),
            )
        )
    return results


async def get_qdrant_client() -> AsyncQdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        _qdrant_client = AsyncQdrantClient(
            url=QDRANT_URL,
            api_key=QDRANT_API_KEY or None,
        )
        print(f"  🔌 Connected to Qdrant at {QDRANT_URL}")
    return _qdrant_client


async def ensure_collection(vector_size: int) -> None:
    """Create the collection with named dense + BM25 sparse vectors (idempotent)."""
    client = await get_qdrant_client()
    collections = await client.get_collections()
    names = [c.name for c in collections.collections]
    if COLLECTION_NAME not in names:
        await client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "dense": models.VectorParams(
                    size=vector_size,
                    distance=models.Distance.COSINE,
                ),
            },
            sparse_vectors_config={
                "bm25": models.SparseVectorParams(
                    modifier=models.Modifier.IDF,
                ),
            },
        )
        print(f"  🆕 Created collection '{COLLECTION_NAME}' (dense dim={vector_size} + BM25)")
    else:
        print(f"  ✅ Collection '{COLLECTION_NAME}' already exists")


async def upsert_points(
    ids: list[str],
    vectors: list[list[float]],
    payloads: list[dict[str, Any]],
) -> None:
    """Upsert points with named dense + BM25 sparse vectors."""
    client = await get_qdrant_client()

    # Prefer heading-enriched text for BM25 (same as vectorstore.py)
    texts = [pay.get("enriched_text", pay["text"]) for pay in payloads]
    sparse_vectors = _encode_sparse(texts)

    points = [
        models.PointStruct(
            id=uid,
            vector={"dense": vec, "bm25": sparse},
            payload=pay,
        )
        for uid, vec, sparse, pay in zip(ids, vectors, sparse_vectors, payloads)
    ]
    await client.upsert(collection_name=COLLECTION_NAME, points=points)

print("✅ Vectorstore client ready")

In [ ]:
"""─────────────────────────────────────────────────────────────────
Upserter — mirrors app/ingestion/upserter.py
Adds time.sleep(SLEEP_BETWEEN_BATCHES) between batches to stay
within ngrok free-tier rate limit (40 connections/min).
─────────────────────────────────────────────────────────────────"""
from __future__ import annotations
import asyncio
import time


def _enrich_text(chunk: Chunk) -> str:
    """Prepend the last heading to the chunk text for richer embeddings."""
    if chunk.headings:
        last_heading = chunk.headings[-1].strip()
        if last_heading:
            return f"{last_heading}\n{chunk.text}"
    return chunk.text


async def upsert_chunks(
    chunks: list[Chunk],
    doc_id: str,
    filename: str,
    batch_size: int = UPSERT_BATCH_SIZE,
    sleep_secs: float = SLEEP_BETWEEN_BATCHES,
) -> int:
    """Embed and upsert chunks in batches. Returns total points upserted."""
    if not chunks:
        return 0

    total = 0
    num_batches = (len(chunks) + batch_size - 1) // batch_size

    for batch_num, start in enumerate(range(0, len(chunks), batch_size), 1):
        batch = chunks[start : start + batch_size]
        print(f"  🔄 Batch {batch_num}/{num_batches} — {len(batch)} chunks")

        # Enrich each chunk's text with its last heading
        enriched_texts = [_enrich_text(c) for c in batch]

        # Dense embeddings via OpenRouter (Qwen3-Embedding-8B)
        vectors = await embed_texts(enriched_texts)

        # On the very first batch, ensure the collection exists
        if start == 0:
            await ensure_collection(vector_size=len(vectors[0]))

        # Build payloads
        ids = [generate_point_id() for _ in batch]
        payloads = [
            {
                "text": c.text,
                "enriched_text": enriched,
                "doc_id": doc_id,
                "filename": filename,
                "chunk_index": c.chunk_index,
                "headings": c.headings,
                "page": c.page,
            }
            for c, enriched in zip(batch, enriched_texts)
        ]

        # Upsert: BM25 sparse + dense named vectors
        await upsert_points(ids=ids, vectors=vectors, payloads=payloads)
        total += len(batch)
        print(f"  ✅ Upserted {total}/{len(chunks)} chunks")

        # Rate-limit: avoid exceeding ngrok free-tier 40 connections/min
        if start + batch_size < len(chunks):
            print(f"  ⏳ Sleeping {sleep_secs}s (ngrok rate limit guard)…")
            time.sleep(sleep_secs)

    print(f"  🎉 Done — upserted {total} chunks for '{filename}'")
    return total

print("✅ Upserter ready")

## 🚀 Cell 4 — Upload PDFs & Run Ingestion

Upload one or more PDF files using the file picker below. Each file will be parsed → chunked → embedded → upserted into your local Qdrant collection.

In [ ]:
from google.colab import files
import asyncio
from pathlib import Path
import time

print("📎 Select one or more PDF files to upload…")
uploaded = files.upload()  # Opens Colab file picker

if not uploaded:
    print("⚠️  No files uploaded. Re-run this cell and select at least one PDF.")
else:
    print(f"\n📥 {len(uploaded)} file(s) uploaded: {list(uploaded.keys())}")

In [ ]:
"""Run the full ingestion pipeline for all uploaded PDFs."""

async def ingest_document(file_path: str | Path) -> dict:
    """Parse → chunk → embed → upsert a single document."""
    file_path = Path(file_path)
    filename = file_path.name
    doc_id = generate_doc_id(filename)

    print(f"\n{'='*60}")
    print(f"📄 Ingesting: {filename}  (doc_id={doc_id})")
    print(f"{'='*60}")
    t0 = time.time()

    # 1. Parse with Docling (GPU-accelerated)
    doc = parse_document(file_path)

    # 2. Chunk with HybridChunker
    chunks = chunk_document(doc)
    if not chunks:
        print(f"  ⚠️  No chunks produced for '{filename}'")
        return {"doc_id": doc_id, "filename": filename, "chunks_count": 0}

    # 3. Embed + Upsert (dense Qwen + sparse BM25)
    count = await upsert_chunks(chunks, doc_id=doc_id, filename=filename)

    elapsed = time.time() - t0
    print(f"\n  ⏱️  Ingestion complete in {elapsed:.1f}s — {count} chunks")
    return {"doc_id": doc_id, "filename": filename, "chunks_count": count}


# ── Run ingestion for every uploaded file ─────────────────────────────────
results = []
for filename in uploaded.keys():
    result = await ingest_document(filename)
    results.append(result)

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 INGESTION SUMMARY")
print("="*60)
total_chunks = 0
for r in results:
    status = "✅" if r["chunks_count"] > 0 else "⚠️ "
    print(f"  {status} {r['filename']:40s} → {r['chunks_count']:>4d} chunks  (doc_id={r['doc_id']})")
    total_chunks += r["chunks_count"]
print(f"\n  Total: {total_chunks} chunks upserted into '{COLLECTION_NAME}'")
print(f"  Qdrant: {QDRANT_URL}")

## 🔍 Cell 5 — Verify (Optional)

Quick sanity check: count points in the collection.

In [ ]:
"""Verify the collection point count via Qdrant."""
client = await get_qdrant_client()
info = await client.get_collection(COLLECTION_NAME)

print(f"\n📦 Collection: {COLLECTION_NAME}")
print(f"   Points count : {info.points_count}")
print(f"   Vectors count: {info.vectors_count}")
print(f"   Status       : {info.status}")